# Notebook 09 — VGG16 Training on RA + MURA Combined Dataset

**Project:** P63 – Multimodal Deep Learning for Autoimmune Disease Diagnosis  
**Phase:** Binary RA vs Control Classification — VGG16 Baseline

---

## Purpose

Train a **VGG16** binary classifier on the same combined balanced dataset used for ResNet-18:

- **Class 1 (RA):** 1120 RA images from RAM-H1200-v1 dataset (`isRA==1` only)
- **Class 0 (Control):** 1120 MURA-negative normal hand X-rays

**Total:** 2240 images (Train=1438, Val=272, Test=530)

---

## Key Points

- Uses **EXISTING combined manifests** from ResNet-18 experiment
- Uses **ImageNet-pretrained VGG16** from torchvision
- Uses **same hyperparameters** as ResNet-18 for fair comparison
- Saves VGG16 checkpoint to `models/ra_mura_vgg16/vgg16_best.pth`
- Does **NOT** modify ResNet-18 checkpoint or any existing files
- Compares VGG16 vs ResNet-18 performance at the end

---

## Important Notes

- MURA-negative samples are **radiographically normal controls** — NOT clinically confirmed absence of RA
- The 80 RAM-H1200 `isRA=0` images are **excluded** from this experiment
- All metrics come from actual model predictions (no fabricated values)
- Previous ResNet-18 near-perfect performance may not represent genuine clinical-level accuracy

---
## Section 1 — Project Setup

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT: {PROJECT_ROOT}')

PROJECT_ROOT: C:\Users\varsh\OneDrive\Documents\FYP


---
## Section 2 — Imports

In [4]:
import os
import json
import random
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models

sns.set_theme(style='whitegrid', palette='muted')
Image.MAX_IMAGE_PIXELS = None

print('All imports successful.')

All imports successful.


---
## Section 3 — Reproducibility and Configuration

In [5]:
# ── Paths ─────────────────────────────────────────────────────────────────────
COMBINED_MAN_DIR = PROJECT_ROOT / 'outputs' / 'manifests' / 'ra_mura_combined'
PLOTS_DIR        = PROJECT_ROOT / 'outputs' / 'plots'
METRICS_DIR      = PROJECT_ROOT / 'outputs' / 'metrics' / 'ra_mura_vgg16'
MODEL_SAVE_DIR   = PROJECT_ROOT / 'models' / 'ra_mura_vgg16'

# ResNet-18 baseline metrics (for comparison)
RESNET18_METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics' / 'ra_mura_combined'

# Create output directories
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────────────
RANDOM_SEED  = 42
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 16
NUM_EPOCHS   = 20
LR           = 0.0001
WEIGHT_DECAY = 0.0001
PATIENCE     = 7
DROPOUT      = 0.5

# ImageNet normalization (standard for pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Expected counts (MUST match)
EXPECTED_COUNTS = {
    'train': {'ra': 719, 'control': 719, 'total': 1438},
    'val'  : {'ra': 136, 'control': 136, 'total': 272},
    'test' : {'ra': 265, 'control': 265, 'total': 530},
}

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Running on CPU (training will be slower)')

print()
print(f'Seed: {RANDOM_SEED}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Learning rate: {LR}')
print(f'Max epochs: {NUM_EPOCHS}')
print(f'Early stopping patience: {PATIENCE}')
print(f'Dropout: {DROPOUT}')

Device: cpu
Running on CPU (training will be slower)

Seed: 42
Batch size: 16
Learning rate: 0.0001
Max epochs: 20
Early stopping patience: 7
Dropout: 0.5


---
## Section 4 — Load Combined Manifests

In [6]:
print('=' * 70)
print('LOADING COMBINED MANIFESTS')
print('=' * 70)

train_csv = COMBINED_MAN_DIR / 'train_combined_manifest.csv'
val_csv   = COMBINED_MAN_DIR / 'val_combined_manifest.csv'
test_csv  = COMBINED_MAN_DIR / 'test_combined_manifest.csv'

# Verify files exist
for path in [train_csv, val_csv, test_csv]:
    if not path.exists():
        raise FileNotFoundError(f'Combined manifest not found: {path}')
    print(f'✓ Found: {path.name}')

# Load manifests
combined_train = pd.read_csv(train_csv)
combined_val   = pd.read_csv(val_csv)
combined_test  = pd.read_csv(test_csv)

print()
print(f'Loaded manifests:')
print(f'  Train: {len(combined_train)} rows')
print(f'  Val  : {len(combined_val)} rows')
print(f'  Test : {len(combined_test)} rows')
print(f'  Total: {len(combined_train) + len(combined_val) + len(combined_test)} rows')

LOADING COMBINED MANIFESTS
✓ Found: train_combined_manifest.csv
✓ Found: val_combined_manifest.csv
✓ Found: test_combined_manifest.csv

Loaded manifests:
  Train: 1438 rows
  Val  : 272 rows
  Test : 530 rows
  Total: 2240 rows


---
## Section 5 — Dataset Verification

In [7]:
print()
print('=' * 70)
print('DATASET VERIFICATION')
print('=' * 70)
print()

# ── 1. Class counts verification ──
print('1. CLASS COUNTS VERIFICATION')
print('-' * 70)

all_ok = True
for split_name, df in [('train', combined_train), ('val', combined_val), ('test', combined_test)]:
    exp = EXPECTED_COUNTS[split_name]
    n_ra      = int((df['label'] == 1).sum())
    n_control = int((df['label'] == 0).sum())
    n_tot     = len(df)
    
    status_ra  = 'PASS' if n_ra == exp['ra'] else 'FAIL'
    status_ctl = 'PASS' if n_control == exp['control'] else 'FAIL'
    status_tot = 'PASS' if n_tot == exp['total'] else 'FAIL'
    
    if status_ra == 'FAIL' or status_ctl == 'FAIL' or status_tot == 'FAIL':
        all_ok = False
    
    print(f'{split_name.upper():<8}:')
    print(f'  RA      : {n_ra:4d} (expected {exp["ra"]:4d}) [{status_ra}]')
    print(f'  Control : {n_control:4d} (expected {exp["control"]:4d}) [{status_ctl}]')
    print(f'  Total   : {n_tot:4d} (expected {exp["total"]:4d}) [{status_tot}]')
    print()

if not all_ok:
    raise AssertionError('❌ Count mismatch detected! STOPPING.')
print('✓ All class counts PASSED.')
print()

# ── 2-4. Source counts ──
print('2-4. SOURCE COUNTS')
print('-' * 70)
combined_all = pd.concat([combined_train, combined_val, combined_test], ignore_index=True)
source_counts = combined_all['source'].value_counts()
print(source_counts)
print()

# ── 5. Label validity ──
print('5. LABEL VALIDITY')
print('-' * 70)
ra_rows = combined_all[combined_all['source'] == 'RAM-H1200_RA']
mura_rows = combined_all[combined_all['source'] == 'MURA_negative_control']

assert (ra_rows['label'] == 1).all(), '❌ RA label error'
print(f'✓ All RA rows have label=1 ({len(ra_rows)} images)')

assert (mura_rows['label'] == 0).all(), '❌ MURA label error'
print(f'✓ All MURA rows have label=0 ({len(mura_rows)} images)')
print()

# ── 6. Missing files ──
print('6. MISSING FILES CHECK')
print('-' * 70)
n_missing = 0
missing_files = []
for _, row in combined_all.iterrows():
    if not Path(row['full_path']).exists():
        n_missing += 1
        if n_missing <= 5:
            missing_files.append(row['full_path'])

if n_missing > 0:
    print(f'❌ MISSING: {n_missing} files')
    for f in missing_files:
        print(f'  {f}')
    raise AssertionError(f'{n_missing} files missing')
else:
    print(f'✓ All files exist (0 missing)')
print()

# ── 7-8. Duplicate checks ──
print('7-8. DUPLICATE CHECKS')
print('-' * 70)
dup_paths = len(combined_all) - combined_all['full_path'].nunique()
dup_fnames = len(combined_all) - combined_all['filename'].nunique()

print(f'Duplicate full_path: {dup_paths} (must be 0)')
print(f'Duplicate filename : {dup_fnames} (must be 0)')

if dup_paths > 0 or dup_fnames > 0:
    raise AssertionError('❌ Duplicate images detected')
print('✓ No duplicates')
print()

# ── 10. No RAM-H1200 isRA=0 ──
print('10. RAM-H1200 isRA=0 EXCLUSION CHECK')
print('-' * 70)
# Since combined manifest only has RA from isRA==1, we verify source counts
expected_ra_total = 1120  # 719+136+265
actual_ra_total = (combined_all['source'] == 'RAM-H1200_RA').sum()
print(f'Total RA images: {actual_ra_total} (expected {expected_ra_total})')
if actual_ra_total != expected_ra_total:
    print('❌ WARNING: RA count mismatch!')
else:
    print('✓ Correct RA count (isRA==1 only)')
print()

# ── 11. All MURA are negative studies ──
print('11. MURA NEGATIVE STUDY VERIFICATION')
print('-' * 70)
all_negative = mura_rows['original_path'].str.lower().str.contains('negative').all()
if not all_negative:
    raise AssertionError('❌ Some MURA controls are not from negative studies')
print(f'✓ All {len(mura_rows)} MURA controls are from negative studies')
print()

# ── 12. Patient-level leakage ──
print('12. PATIENT-LEVEL LEAKAGE CHECK')
print('-' * 70)

# MURA patient leakage
mura_train_pts = set(combined_train[combined_train['source']=='MURA_negative_control']['patient_id'])
mura_val_pts   = set(combined_val[combined_val['source']=='MURA_negative_control']['patient_id'])
mura_test_pts  = set(combined_test[combined_test['source']=='MURA_negative_control']['patient_id'])

overlap_val_test  = mura_val_pts & mura_test_pts
overlap_val_train = mura_val_pts & mura_train_pts
overlap_test_train = mura_test_pts & mura_train_pts

print(f'MURA patients:')
print(f'  Train: {len(mura_train_pts)} unique')
print(f'  Val  : {len(mura_val_pts)} unique')
print(f'  Test : {len(mura_test_pts)} unique')
print(f'  Val ∩ Test  : {len(overlap_val_test)} (must be 0)')
print(f'  Val ∩ Train : {len(overlap_val_train)} (must be 0)')
print(f'  Test ∩ Train: {len(overlap_test_train)} (must be 0)')

if overlap_val_test or overlap_val_train or overlap_test_train:
    raise AssertionError('❌ MURA patient leakage detected!')
print('✓ Zero MURA patient overlap')
print()

print('=' * 70)
print('✓✓✓ ALL VERIFICATION CHECKS PASSED ✓✓✓')
print('=' * 70)


DATASET VERIFICATION

1. CLASS COUNTS VERIFICATION
----------------------------------------------------------------------
TRAIN   :
  RA      :  719 (expected  719) [PASS]
  Control :  719 (expected  719) [PASS]
  Total   : 1438 (expected 1438) [PASS]

VAL     :
  RA      :  136 (expected  136) [PASS]
  Control :  136 (expected  136) [PASS]
  Total   :  272 (expected  272) [PASS]

TEST    :
  RA      :  265 (expected  265) [PASS]
  Control :  265 (expected  265) [PASS]
  Total   :  530 (expected  530) [PASS]

✓ All class counts PASSED.

2-4. SOURCE COUNTS
----------------------------------------------------------------------
source
RAM-H1200_RA             1120
MURA_negative_control    1120
Name: count, dtype: int64

5. LABEL VALIDITY
----------------------------------------------------------------------
✓ All RA rows have label=1 (1120 images)
✓ All MURA rows have label=0 (1120 images)

6. MISSING FILES CHECK
----------------------------------------------------------------------
✓ Al

---
## Section 6 — Image Preprocessing and Transforms

In [8]:
print('Defining image transforms for VGG16...')
print()

def get_transforms(split, img_size, mean, std):
    """
    Return transform pipeline for VGG16.
    
    Training: Augmentation + resize + normalize
    Val/Test: Resize + normalize only
    """
    normalize = T.Normalize(mean=mean, std=std)
    
    if split == 'train':
        return T.Compose([
            T.Resize(img_size),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(10),
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            T.ToTensor(),
            normalize,
        ])
    else:
        return T.Compose([
            T.Resize(img_size),
            T.ToTensor(),
            normalize,
        ])

train_transform = get_transforms('train', IMG_SIZE, IMAGENET_MEAN, IMAGENET_STD)
val_transform   = get_transforms('val',   IMG_SIZE, IMAGENET_MEAN, IMAGENET_STD)
test_transform  = get_transforms('test',  IMG_SIZE, IMAGENET_MEAN, IMAGENET_STD)

print('Transforms defined:')
print(f'  Input size: {IMG_SIZE}')
print(f'  Normalization: ImageNet (mean={IMAGENET_MEAN}, std={IMAGENET_STD})')
print(f'  Training augmentation: HFlip, Rotation(±10°), ColorJitter, Translate')
print(f'  Val/Test: Resize only (no augmentation)')

Defining image transforms for VGG16...

Transforms defined:
  Input size: (224, 224)
  Normalization: ImageNet (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  Training augmentation: HFlip, Rotation(±10°), ColorJitter, Translate
  Val/Test: Resize only (no augmentation)


---
## Section 7 — Dataset and DataLoader Creation

In [9]:
class CombinedRAMURADataset(Dataset):
    """
    PyTorch Dataset for combined RA + MURA control images.
    
    Reads manifest CSV and loads images on-the-fly.
    Converts all images to RGB (3-channel) for VGG16.
    """
    def __init__(self, manifest_df, transform=None):
        self.manifest  = manifest_df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.manifest)
    
    def __getitem__(self, idx):
        row   = self.manifest.iloc[idx]
        path  = Path(row['full_path'])
        label = int(row['label'])
        
        # Load and convert to RGB
        img = Image.open(path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
        
        return img, label

# Create datasets
train_dataset = CombinedRAMURADataset(combined_train, transform=train_transform)
val_dataset   = CombinedRAMURADataset(combined_val,   transform=val_transform)
test_dataset  = CombinedRAMURADataset(combined_test,  transform=test_transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Datasets created:')
print(f'  Train: {len(train_dataset)} images, {len(train_loader)} batches')
print(f'  Val  : {len(val_dataset)} images, {len(val_loader)} batches')
print(f'  Test : {len(test_dataset)} images, {len(test_loader)} batches')

Datasets created:
  Train: 1438 images, 90 batches
  Val  : 272 images, 17 batches
  Test : 530 images, 34 batches


---
## Section 8 — VGG16 Model Definition

In [10]:
print('=' * 70)
print('DEFINING VGG16 MODEL')
print('=' * 70)
print()

class BinaryVGG16(nn.Module):
    """
    VGG16 binary classifier.
    
    Uses ImageNet-pretrained VGG16 backbone.
    Replaces final classifier with:
      - Linear(4096 → 512)
      - ReLU
      - Dropout
      - Linear(512 → 1) for binary classification
    """
    def __init__(self, dropout=0.5, pretrained=True):
        super().__init__()
        
        # Load pretrained VGG16
        self.backbone = models.vgg16(pretrained=pretrained)
        
        # VGG16 classifier has 3 Linear layers:
        # classifier[0]: Linear(25088 → 4096)
        # classifier[3]: Linear(4096 → 4096)
        # classifier[6]: Linear(4096 → 1000)
        
        # Replace final classifier layer for binary classification
        in_features = self.backbone.classifier[6].in_features  # 4096
        
        self.backbone.classifier[6] = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
        )
    
    def forward(self, x):
        return self.backbone(x)

# Create model
model = BinaryVGG16(dropout=DROPOUT, pretrained=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model: VGG16')
print(f'Pretrained: ImageNet')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print()
print('Model architecture:')
print(model)

DEFINING VGG16 MODEL

Model: VGG16
Pretrained: ImageNet
Total parameters: 136,358,721
Trainable parameters: 136,358,721

Model architecture:
BinaryVGG16(
  (backbone): VGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      

---
## Section 9 — Training Configuration

In [11]:
print()

print('=' * 70)
print('TRAINING CONFIGURATION')
print('=' * 70)

# Loss function
criterion = nn.BCEWithLogitsLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

print('Loss function: BCEWithLogitsLoss')
print(f'Optimizer: Adam (lr={LR}, weight_decay={WEIGHT_DECAY})')
print('Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)')
print(f'Early stopping patience: {PATIENCE} epochs')
print(f'Maximum epochs: {NUM_EPOCHS}')


TRAINING CONFIGURATION
Loss function: BCEWithLogitsLoss
Optimizer: Adam (lr=0.0001, weight_decay=0.0001)
Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)
Early stopping patience: 7 epochs
Maximum epochs: 20


---
## Section 10 — Training Loop

In [12]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

print('Training functions defined.')

Training functions defined.


In [ ]:
print()
print('=' * 70)
print('TRAINING VGG16')
print('=' * 70)
print()

best_val_loss = float('inf')
patience_counter = 0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    # Print progress
    print(f'Epoch {epoch:2d}/{NUM_EPOCHS}  '
          f'Train: Loss={train_loss:.4f} Acc={train_acc:.4f}  '
          f'Val: Loss={val_loss:.4f} Acc={val_acc:.4f}  '
          f'LR={optimizer.param_groups[0]["lr"]:.2e}')
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_DIR / 'vgg16_best.pth')
        print(f'  → Best model saved (val_loss={val_loss:.4f})')
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print()
        print(f'Early stopping triggered at epoch {epoch}')
        print(f'Best val_loss: {best_val_loss:.4f}')
        break

print()
print('=' * 70)
print(f'TRAINING COMPLETE')
print(f'Best validation loss: {best_val_loss:.4f}')
print(f'Model saved to: {MODEL_SAVE_DIR / "vgg16_best.pth"}')
print('=' * 70)


TRAINING VGG16



---
## Section 11 — Training Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 's-', label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], 'o-', label='Train', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'], 's-', label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(epochs_range, history['lr'], 'o-', color='orange', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '09_vgg16_training_history.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Training curves saved to: {PLOTS_DIR / "09_vgg16_training_history.png"}')

---
## Section 12 — Load Best Checkpoint

In [ ]:
print('=' * 70)
print('LOADING BEST CHECKPOINT')
print('=' * 70)

checkpoint_path = MODEL_SAVE_DIR / 'vgg16_best.pth'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

print(f'✓ Loaded best checkpoint from: {checkpoint_path}')
print(f'  Best validation loss: {best_val_loss:.4f}')

---
## Section 13 — Test Set Evaluation

In [ ]:
print()
print('=' * 70)
print('TEST SET EVALUATION')
print('=' * 70)
print()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        
        outputs = model(imgs)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        all_labels.append(labels.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

# Concatenate results
all_labels = np.concatenate(all_labels).flatten()
all_preds = np.concatenate(all_preds).flatten()
all_probs = np.concatenate(all_probs).flatten()

# Calculate metrics
test_acc  = accuracy_score(all_labels, all_preds)
test_prec = precision_score(all_labels, all_preds, zero_division=0)
test_rec  = recall_score(all_labels, all_preds, zero_division=0)
test_f1   = f1_score(all_labels, all_preds, zero_division=0)
test_auc  = roc_auc_score(all_labels, all_probs)

print('VGG16 Test Set Performance:')
print('-' * 70)
print(f'Accuracy  : {test_acc:.4f}')
print(f'Precision : {test_prec:.4f}')
print(f'Recall    : {test_rec:.4f}')
print(f'F1-Score  : {test_f1:.4f}')
print(f'ROC-AUC   : {test_auc:.4f}')
print()
print('Classification Report:')
print('-' * 70)
print(classification_report(all_labels, all_preds, 
                          target_names=['MURA Control (0)', 'RA (1)'],
                          zero_division=0))

---
## Section 14 — Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['MURA Control (0)', 'RA (1)'],
            yticklabels=['MURA Control (0)', 'RA (1)'],
            ax=ax, annot_kws={'size': 16})
ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label', fontsize=13, fontweight='bold')
ax.set_title('VGG16 Confusion Matrix — Test Set', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '09_vgg16_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Confusion matrix saved to: {PLOTS_DIR / "09_vgg16_confusion_matrix.png"}')

---
## Section 15 — ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, lw=2.5, label=f'VGG16 (AUC = {test_auc:.4f})', color='#2E86AB')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier', alpha=0.5)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=13, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=13, fontweight='bold')
ax.set_title('VGG16 ROC Curve — Test Set', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '09_vgg16_roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'ROC curve saved to: {PLOTS_DIR / "09_vgg16_roc_curve.png"}')

---
## Section 16 — Sample Test Predictions

In [ ]:
# Add predictions to test dataframe
test_results = combined_test.copy()
test_results['pred_label'] = all_preds.astype(int)
test_results['pred_prob'] = all_probs

# Sample correct and incorrect predictions
correct_preds = test_results[test_results['label'] == test_results['pred_label']]
incorrect_preds = test_results[test_results['label'] != test_results['pred_label']]

n_correct_samples = min(4, len(correct_preds))
n_incorrect_samples = min(4, len(incorrect_preds))

correct_samples = correct_preds.sample(n_correct_samples, random_state=42) if n_correct_samples > 0 else pd.DataFrame()
incorrect_samples = incorrect_preds.sample(n_incorrect_samples, random_state=42) if n_incorrect_samples > 0 else pd.DataFrame()

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('VGG16 Sample Test Predictions', fontsize=16, fontweight='bold')

# Correct predictions
for i in range(4):
    if i < len(correct_samples):
        row = correct_samples.iloc[i]
        with Image.open(row['full_path']) as img:
            thumb = img.convert('RGB') if row['source'] == 'RAM-H1200_RA' else img.copy()
            thumb.thumbnail((300, 300))
        
        axes[0][i].imshow(thumb)
        axes[0][i].set_title(
            f'✓ CORRECT
True: {row["label"]} | Pred: {row["pred_label"]}
' +
            f'Prob: {row["pred_prob"]:.3f}
{row["filename"][:20]}...',
            fontsize=9, color='green', fontweight='bold'
        )
    else:
        axes[0][i].text(0.5, 0.5, 'No sample', ha='center', va='center', fontsize=12)
    axes[0][i].axis('off')

# Incorrect predictions
for i in range(4):
    if i < len(incorrect_samples):
        row = incorrect_samples.iloc[i]
        with Image.open(row['full_path']) as img:
            thumb = img.convert('RGB') if row['source'] == 'RAM-H1200_RA' else img.copy()
            thumb.thumbnail((300, 300))
        
        axes[1][i].imshow(thumb)
        axes[1][i].set_title(
            f'✗ INCORRECT
True: {row["label"]} | Pred: {row["pred_label"]}
' +
            f'Prob: {row["pred_prob"]:.3f}
{row["filename"][:20]}...',
            fontsize=9, color='red', fontweight='bold'
        )
    else:
        axes[1][i].text(0.5, 0.5, 'No sample', ha='center', va='center', fontsize=12)
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '09_vgg16_sample_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Sample predictions saved to: {PLOTS_DIR / "09_vgg16_sample_predictions.png"}')
print()
print(f'Correct predictions: {len(correct_preds)} / {len(test_results)} ({len(correct_preds)/len(test_results)*100:.2f}%)')
print(f'Incorrect predictions: {len(incorrect_preds)} / {len(test_results)} ({len(incorrect_preds)/len(test_results)*100:.2f}%)')

---
## Section 17 — Save VGG16 Metrics

In [ ]:
print()
print('=' * 70)
print('SAVING VGG16 METRICS')
print('=' * 70)

# Metrics dictionary
vgg16_metrics = {
    'model': 'VGG16',
    'architecture': 'torchvision.models.vgg16 (ImageNet pretrained)',
    'dataset': 'RA + MURA combined',
    'train_images': len(combined_train),
    'val_images': len(combined_val),
    'test_images': len(combined_test),
    'total_images': len(combined_all),
    'seed': RANDOM_SEED,
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'weight_decay': WEIGHT_DECAY,
    'dropout': DROPOUT,
    'max_epochs': NUM_EPOCHS,
    'early_stopping_patience': PATIENCE,
    'epochs_trained': len(history['train_loss']),
    'best_val_loss': float(best_val_loss),
    'test_accuracy': float(test_acc),
    'test_precision': float(test_prec),
    'test_recall': float(test_rec),
    'test_f1': float(test_f1),
    'test_roc_auc': float(test_auc),
    'confusion_matrix': cm.tolist(),
}

# Save as JSON
metrics_json_path = METRICS_DIR / 'vgg16_test_metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(vgg16_metrics, f, indent=2)
print(f'✓ Saved: {metrics_json_path}')

# Save as CSV
metrics_csv_path = METRICS_DIR / 'vgg16_test_metrics.csv'
metrics_df = pd.DataFrame([{
    'Model': 'VGG16',
    'Accuracy': test_acc,
    'Precision': test_prec,
    'Recall': test_rec,
    'F1': test_f1,
    'ROC-AUC': test_auc,
}])
metrics_df.to_csv(metrics_csv_path, index=False)
print(f'✓ Saved: {metrics_csv_path}')

# Save training history
history_csv_path = METRICS_DIR / 'vgg16_training_history.csv'
history_df = pd.DataFrame({
    'epoch': list(range(1, len(history['train_loss']) + 1)),
    'train_loss': history['train_loss'],
    'train_acc': history['train_acc'],
    'val_loss': history['val_loss'],
    'val_acc': history['val_acc'],
    'lr': history['lr'],
})
history_df.to_csv(history_csv_path, index=False)
print(f'✓ Saved: {history_csv_path}')
print()
print(f'All VGG16 metrics saved to: {METRICS_DIR}')

---
## Section 18 — Compare with ResNet-18 Baseline

In [ ]:
print()
print('=' * 70)
print('VGG16 vs RESNET-18 COMPARISON')
print('=' * 70)
print()

# Try to load ResNet-18 metrics
resnet18_metrics_file = RESNET18_METRICS_DIR / 'resnet18_test_metrics.json'

if resnet18_metrics_file.exists():
    print(f'✓ Found ResNet-18 metrics: {resnet18_metrics_file}')
    with open(resnet18_metrics_file, 'r') as f:
        resnet18_metrics = json.load(f)
    
    # Extract ResNet-18 test metrics
    r18_acc = resnet18_metrics.get('test_accuracy', None)
    r18_prec = resnet18_metrics.get('test_precision', None)
    r18_rec = resnet18_metrics.get('test_recall', None)
    r18_f1 = resnet18_metrics.get('test_f1', None)
    r18_auc = resnet18_metrics.get('test_roc_auc', None)
    
    # Create comparison dataframe
    comparison_df = pd.DataFrame([
        {
            'Model': 'ResNet-18',
            'Accuracy': r18_acc,
            'Precision': r18_prec,
            'Recall': r18_rec,
            'F1-Score': r18_f1,
            'ROC-AUC': r18_auc,
        },
        {
            'Model': 'VGG16',
            'Accuracy': test_acc,
            'Precision': test_prec,
            'Recall': test_rec,
            'F1-Score': test_f1,
            'ROC-AUC': test_auc,
        },
    ])
    
    print()
    print('Model Comparison Table:')
    print('-' * 70)
    print(comparison_df.to_string(index=False))
    print()
    
    # Save comparison
    comparison_csv = METRICS_DIR / 'vgg16_vs_resnet18_comparison.csv'
    comparison_df.to_csv(comparison_csv, index=False)
    print(f'✓ Comparison saved to: {comparison_csv}')
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
    x = np.arange(len(metrics_to_plot))
    width = 0.35
    
    resnet18_values = [r18_acc, r18_prec, r18_rec, r18_f1, r18_auc]
    vgg16_values = [test_acc, test_prec, test_rec, test_f1, test_auc]
    
    bars1 = ax.bar(x - width/2, resnet18_values, width, label='ResNet-18', color='#FF6B6B', alpha=0.8)
    bars2 = ax.bar(x + width/2, vgg16_values, width, label='VGG16', color='#4ECDC4', alpha=0.8)
    
    ax.set_ylabel('Score', fontsize=13, fontweight='bold')
    ax.set_xlabel('Metric', fontsize=13, fontweight='bold')
    ax.set_title('VGG16 vs ResNet-18 — Test Set Performance', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_to_plot, fontsize=11)
    ax.legend(fontsize=12)
    ax.set_ylim([0, 1.1])
    ax.grid(True, axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '09_vgg16_vs_resnet18_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    print(f'✓ Comparison plot saved to: {PLOTS_DIR / "09_vgg16_vs_resnet18_comparison.png"}')
    
else:
    print('⚠ ResNet-18 metrics not found.')
    print(f'  Expected location: {resnet18_metrics_file}')
    print()
    print('VGG16 standalone results:')
    print('-' * 70)
    print(f'  Accuracy : {test_acc:.4f}')
    print(f'  Precision: {test_prec:.4f}')
    print(f'  Recall   : {test_rec:.4f}')
    print(f'  F1-Score : {test_f1:.4f}')
    print(f'  ROC-AUC  : {test_auc:.4f}')
    print()
    print('Run notebook 08 (ResNet-18) and save metrics to enable comparison.')

---
## Section 19 — Final Experiment Summary

In [ ]:
print()
print('=' * 70)
print('FINAL EXPERIMENT SUMMARY — VGG16')
print('=' * 70)
print()
print('Model: VGG16 (ImageNet pretrained)')
print('Task: Binary classification (RA vs MURA-negative control)')
print()
print('Dataset:')
print(f'  Train images     : {len(combined_train)} (RA={int((combined_train["label"]==1).sum())}, Control={int((combined_train["label"]==0).sum())})')
print(f'  Validation images: {len(combined_val)} (RA={int((combined_val["label"]==1).sum())}, Control={int((combined_val["label"]==0).sum())})')
print(f'  Test images      : {len(combined_test)} (RA={int((combined_test["label"]==1).sum())}, Control={int((combined_test["label"]==0).sum())})')
print(f'  Total images     : {len(combined_all)}')
print()
print('Training Configuration:')
print(f'  Seed             : {RANDOM_SEED}')
print(f'  Batch size       : {BATCH_SIZE}')
print(f'  Learning rate    : {LR}')
print(f'  Weight decay     : {WEIGHT_DECAY}')
print(f'  Dropout          : {DROPOUT}')
print(f'  Max epochs       : {NUM_EPOCHS}')
print(f'  Epochs trained   : {len(history["train_loss"])}')
print(f'  Early stop patience: {PATIENCE}')
print()
print('Best Model:')
print(f'  Validation loss  : {best_val_loss:.4f}')
print(f'  Checkpoint saved : {MODEL_SAVE_DIR / "vgg16_best.pth"}')
print()
print('Test Set Performance:')
print(f'  Accuracy         : {test_acc:.4f}')
print(f'  Precision        : {test_prec:.4f}')
print(f'  Recall           : {test_rec:.4f}')
print(f'  F1-Score         : {test_f1:.4f}')
print(f'  ROC-AUC          : {test_auc:.4f}')
print()
print('Important Research Notes:')
print('─' * 70)
print('1. MURA-negative samples are used as operational normal/control')
print('   samples and are NOT clinically confirmed absence of RA.')
print()
print('2. High performance metrics (if observed) should be interpreted')
print('   cautiously and do NOT necessarily represent genuine clinical-')
print('   level diagnostic accuracy.')
print()
print('3. The original RAM-H1200 dataset contains 80 isRA=0 control')
print('   images that were EXCLUDED from this experiment.')
print()
print('4. All metrics come from actual model predictions on the test set.')
print('   No values were fabricated or manually entered.')
print()
print('5. Fair comparison with ResNet-18 is enabled by using:')
print('   - Same combined train/val/test splits')
print('   - Same seed, batch size, learning rate, training config')
print('   - Same data augmentation strategy')
print()
print('=' * 70)
print('✓✓✓ EXPERIMENT COMPLETE ✓✓✓')
print('=' * 70)